# Function definitions

In [ ]:
import numpy as np
import pandas as pd
import duckdb
from matplotlib import pyplot as plt

def inspect_converted_episode(
    parquet_path,
    lerobot_path,
    episode_index=0,
    start=0,
    end=-1,
    joint_labels=None,
):
    """
    Load and inspect one source-Parquet / LeRobot episode pair.

    The source episode is selected from successful source UUIDs ordered by UUID,
    matching the converter's normal successful-episode ordering.

    Layout:
      row 1: state comparison | action comparison
      row 2: all values combined
      row 3: Parquet obs/action | LeRobot obs/action

    Solid lines = observations/states
    Dotted lines = actions

    `start` is inclusive; `end` is exclusive. Set end=-1 for all valid rows.
    Row 0 is excluded from both datasets because the source action is NaN there.
    """
    def as_matrix(series):
        return np.vstack([
            np.asarray(value, dtype=np.float32).reshape(-1)
            for value in series.to_numpy()
        ])

    # Escape paths because read_parquet requires a SQL string literal.
    parquet_sql_path = str(parquet_path).replace("'", "''")
    lerobot_sql_path = str(lerobot_path).replace("'", "''")

    con = duckdb.connect()

    # This matches the source-side UUID ordering used for successful episodes.
    parquet_uuids = con.execute(f"""
        SELECT DISTINCT uuid
        FROM read_parquet('{parquet_sql_path}')
        WHERE success
        ORDER BY uuid
    """).fetchnumpy()["uuid"]

    if not 0 <= episode_index < len(parquet_uuids):
        raise IndexError(
            f"episode_index={episode_index} is outside the available "
            f"source episode range 0:{len(parquet_uuids)}"
        )

    uuid = parquet_uuids[episode_index]

    p_df = con.execute("""
        SELECT
            step,
            obs.right.gripper AS obs_gripper,
            action.right.gripper AS action_gripper,
            obs.right.joints AS obs_joints,
            info.right.absolute_action AS absolute_action
        FROM read_parquet(?)
        WHERE uuid = ?
        ORDER BY step
    """, [parquet_path, uuid]).df()

    l_df = con.execute("""
        SELECT
            timestamp,
            frame_index,
            episode_index,
            "observation.state" AS observation_state,
            action
        FROM read_parquet(?)
        WHERE episode_index = ?
        ORDER BY timestamp, frame_index
    """, [lerobot_path, episode_index]).df()

    if p_df.empty:
        raise ValueError(f"No source rows found for UUID {uuid}")
    if l_df.empty:
        raise ValueError(f"No LeRobot rows found for episode_index {episode_index}")

    # Source action at row 0 is NaN. Keep both sources aligned with prior plots.
    p_df = p_df.iloc[1:].reset_index(drop=True)
    l_df = l_df.iloc[1:].reset_index(drop=True)

    p_state = np.hstack([
        as_matrix(p_df["obs_joints"]),
        as_matrix(p_df["obs_gripper"]),
    ])
    p_action = np.hstack([
        as_matrix(p_df["absolute_action"]),
        as_matrix(p_df["action_gripper"]),
    ])
    l_state = as_matrix(l_df["observation_state"])
    l_action = as_matrix(l_df["action"])

    max_length = min(len(p_df), len(l_df))
    stop = max_length if end == -1 else end

    if not (0 <= start < stop <= max_length):
        raise ValueError(
            f"Use 0 <= start < end <= {max_length}; "
            f"end=-1 means {max_length}."
        )

    p_state, p_action = p_state[start:stop], p_action[start:stop]
    l_state, l_action = l_state[start:stop], l_action[start:stop]

    p_x = p_df["step"].to_numpy()[start:stop]
    l_x = l_df["frame_index"].to_numpy()[start:stop]
    aligned_x = np.arange(start, stop)

    num_dims = min(
        p_state.shape[1], p_action.shape[1],
        l_state.shape[1], l_action.shape[1],
    )

    if joint_labels is None:
        joint_labels = {
            0: "joint_1", 1: "joint_2", 2: "joint_3", 3: "joint_4",
            4: "joint_5", 5: "joint_6", 6: "joint_7", 7: "gripper",
        }

    colors = plt.cm.tab10(np.arange(num_dims))

    # 16:9 figure containing five axes in a 2–1–2 arrangement.
    fig = plt.figure(figsize=(16, 9), constrained_layout=True)
    grid = fig.add_gridspec(3, 2, height_ratios=[1, 1.1, 1])

    ax_state = fig.add_subplot(grid[0, 0])
    ax_action = fig.add_subplot(grid[0, 1])
    ax_combined = fig.add_subplot(grid[1, :])
    ax_parquet = fig.add_subplot(grid[2, 0])
    ax_lerobot = fig.add_subplot(grid[2, 1], sharey=ax_parquet)

    for dim in range(num_dims):
        name = joint_labels.get(dim, f"dim_{dim}")
        color = colors[dim]

        # Row 1: direct source-vs-converted comparisons.
        ax_state.plot(p_x, p_state[:, dim], color=color, linewidth=1.2,
                      label=f"{name} — parquet")
        ax_state.plot(l_x, l_state[:, dim], color=color, linewidth=1.2,
                      linestyle=":", label=f"{name} — lerobot")

        ax_action.plot(p_x, p_action[:, dim], color=color, linewidth=1.2,
                       label=f"{name} — parquet")
        ax_action.plot(l_x, l_action[:, dim], color=color, linewidth=1.2,
                       linestyle=":", label=f"{name} — lerobot")

        # Row 2: all sources and signal types on one aligned axis.
        ax_combined.plot(aligned_x, p_state[:, dim], color=color, linewidth=1.2,
                         label=f"{name} P obs")
        ax_combined.plot(aligned_x, p_action[:, dim], color=color, linewidth=1.2,
                         linestyle=":", label=f"{name} P act")
        ax_combined.plot(aligned_x, l_state[:, dim], color=color, linewidth=1.0,
                         alpha=0.5, label=f"{name} L obs")
        ax_combined.plot(aligned_x, l_action[:, dim], color=color, linewidth=1.0,
                         linestyle=":", alpha=0.5, label=f"{name} L act")

        # Row 3: source-specific state/action overlays.
        ax_parquet.plot(aligned_x, p_state[:, dim], color=color, linewidth=1.2,
                        label=f"{name} — obs")
        ax_parquet.plot(aligned_x, p_action[:, dim], color=color, linewidth=1.2,
                        linestyle=":", label=f"{name} — action")

        ax_lerobot.plot(aligned_x, l_state[:, dim], color=color, linewidth=1.2,
                        label=f"{name} — obs")
        ax_lerobot.plot(aligned_x, l_action[:, dim], color=color, linewidth=1.2,
                        linestyle=":", label=f"{name} — action")

    ax_state.set_title("State: Parquet vs LeRobot", fontsize=10)
    ax_action.set_title("Action: Parquet vs LeRobot", fontsize=10)
    ax_combined.set_title("All state and action values", fontsize=10)
    ax_parquet.set_title("Parquet: observation and action", fontsize=10)
    ax_lerobot.set_title("LeRobot: observation and action", fontsize=10)

    for ax in (ax_state, ax_action):
        ax.set_xlabel("Source step / LeRobot frame index", fontsize=8)

    for ax in (ax_combined, ax_parquet, ax_lerobot):
        ax.set_xlabel("Aligned valid-row index", fontsize=8)

    for ax in (ax_state, ax_action, ax_combined, ax_parquet, ax_lerobot):
        ax.grid(alpha=0.3)
        ax.tick_params(labelsize=7)

    ax_state.set_ylabel("State", fontsize=8)
    ax_action.set_ylabel("Action", fontsize=8)
    ax_combined.set_ylabel("Position / target", fontsize=8)
    ax_parquet.set_ylabel("Position / target", fontsize=8)

    ax_state.legend(ncol=2, fontsize=4.5)
    ax_action.legend(ncol=2, fontsize=4.5)
    ax_combined.legend(ncol=8, fontsize=4.2)
    ax_parquet.legend(ncol=2, fontsize=4.5)
    ax_lerobot.legend(ncol=2, fontsize=4.5)

    fig.suptitle(
        f"Episode {episode_index} — UUID {uuid} — valid-row range {start}:{stop}",
        fontsize=13,
    )
    plt.show()

In [ ]:
parquet_path = "/home/bien/Documents/Development/RCS/datasets/dataset_parquet/with_wood_cover/utn_tapedusb_insertion"
lerobot_path = "/home/bien/Documents/Development/RCS/datasets/dataset_lerobot/utn_tapedusb_insertion/data/chunk-000/*.parquet"

inspect_converted_episode(parquet_path,lerobot_path,episode_index=0,start=0,end=-1)
